In [ ]:
import gc
import os

import numpy as np
import pandas as pd
import torch
from chemprop import data, featurizers
from chemprop.nn import (
    BondMessagePassing,
    MeanAggregation,
    SumAggregation,
)
from lightning import pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.callbacks.early_stopping import EarlyStopping
from lightning.pytorch.loggers import TensorBoardLogger
from torch import nn

from benchmarks import BestEpochTracker, MultiTargetScaler, PrintEpochMetricsCallback
from benchmarks.lightning_integration import ChemPropLightningModule

torch.set_float32_matmul_precision("medium")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

k_folds = 5
FOLDS = range(1, k_folds + 1)

target_properties = [
    "xtb_homo",
    "xtb_lumo",
    "xtb_hl_gap",
    "pm7_homo",
    "pm7_lumo",
    "pm7_hl_gap",
    "pm7_hof",
]
property_types = [
    "intensive",
    "intensive",
    "intensive",
    "intensive",
    "intensive",
    "intensive",
    "extensive",
]

TAG = "chemprop_1.1M-graph"
model_path_template = "../models/" + TAG + "-[PAC5-4]-fold_{fold}"
model_logs_dir = "../logs"
model_dataset_dir = "../dataset"
model_logs_template = TAG + "-[PAC5-4]-fold_{fold}"

SPLIT_DIR = model_dataset_dir + "/PAC5-4"
NUM_WORKERS = 0

featurizer = featurizers.SimpleMoleculeMolGraphFeaturizer()

BATCH_SIZE = 256
MAX_EPOCHS = 500
ES_PATIENCE = 50  # Epochs without val_loss improvement before stopping
LR_PATIENCE = 25  # Epochs without val_loss improvement before halving learning rate
MIN_LR = 1e-6
LR = 1e-3


def _plateau_optimizers(self):
    """Configure AdamW with ReduceLROnPlateau in place of OneCycleLR."""
    optimizer = torch.optim.AdamW(
        self.parameters(), lr=self.lr, weight_decay=self.weight_decay
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=LR_PATIENCE, min_lr=MIN_LR
    )
    return {
        "optimizer": optimizer,
        "lr_scheduler": {
            "scheduler": scheduler,
            "monitor": "val_loss",
            "interval": "epoch",
            "frequency": 1,
        },
    }


class PlateauChemPropLightningModule(ChemPropLightningModule):
    """ChemPropLightningModule with ReduceLROnPlateau instead of OneCycleLR."""

    configure_optimizers = _plateau_optimizers


def load_split(fold, part, scaler=None):
    df = pd.read_csv(SPLIT_DIR + f"/fold_{fold}_{part}.csv").dropna(
        subset=target_properties
    )
    smiles = df["smiles"].values
    targets = df[target_properties].values.astype(np.float32)
    if scaler is None:
        scaler = MultiTargetScaler()
        scaler.fit(targets)
    scaled = scaler.transform(targets)
    pts = [
        data.MoleculeDatapoint.from_smi(smi, torch.tensor(y, dtype=torch.float32))
        for smi, y in zip(smiles, scaled)
    ]
    # Chemprop annotates MoleculeDatapoint.from_smi with return type of its mixin class instead of MoleculeDatapoint
    return data.MoleculeDataset(pts, featurizer, n_workers=NUM_WORKERS), scaler  # pyrefly: ignore[bad-argument-type]


D_H = 512
DEPTH = 3
FFN_HIDDEN = 512
FFN_LAYERS = 2


class DualReadoutMPNN(nn.Module):
    """Shared D-MPNN backbone with mean readout for intensive targets and sum readout for extensive targets."""

    def __init__(
        self,
        property_types,
        d_h=D_H,
        depth=DEPTH,
        ffn_hidden=FFN_HIDDEN,
        ffn_layers=FFN_LAYERS,
    ):
        super().__init__()
        self.property_types = list(property_types)
        self.n_int = self.property_types.count("intensive")
        self.n_ext = self.property_types.count("extensive")

        self.mp = BondMessagePassing(d_h=d_h, depth=depth)
        d = self.mp.output_dim

        self.mean_agg = MeanAggregation()
        self.sum_agg = SumAggregation()

        def make_head(n_out):
            layers = [nn.BatchNorm1d(d)]
            in_dim = d
            for _ in range(ffn_layers - 1):
                layers += [
                    nn.Linear(in_dim, ffn_hidden),
                    nn.BatchNorm1d(ffn_hidden),
                    nn.SiLU(),
                ]
                in_dim = ffn_hidden
            layers += [nn.Linear(in_dim, n_out)]
            return nn.Sequential(*layers)

        if self.n_int:
            self.int_head = make_head(self.n_int)
        if self.n_ext:
            self.ext_head = make_head(self.n_ext)

    def forward(self, bmg, V_d=None, X_d=None):
        H = self.mp(bmg, V_d)
        int_pred = None
        ext_pred = None
        if self.n_int:
            int_pred = self.int_head(self.mean_agg(H, bmg.batch))
        if self.n_ext:
            ext_pred = self.ext_head(self.sum_agg(H, bmg.batch))

        out, i, e = [], 0, 0
        for p in self.property_types:
            if p == "intensive":
                if int_pred is None:
                    raise RuntimeError(
                        "The model does not have a readout head for intensive properties."
                    )
                out.append(int_pred[:, i : i + 1])
                i += 1
            else:
                if ext_pred is None:
                    raise RuntimeError(
                        "The model does not have a readout head for extensive properties."
                    )
                out.append(ext_pred[:, e : e + 1])
                e += 1
        return torch.cat(out, dim=1)


def build_model():
    return DualReadoutMPNN(
        property_types=property_types,
        d_h=D_H,
        depth=DEPTH,
        ffn_hidden=FFN_HIDDEN,
        ffn_layers=FFN_LAYERS,
    )


_m = build_model()
print(
    f"Chemprop matched: d_h {D_H}, depth {DEPTH}, ffn {FFN_HIDDEN}x{FFN_LAYERS}, dual readout"
)
print(
    f"Trainable parameters: {sum(p.numel() for p in _m.parameters() if p.requires_grad):,}"
)
del _m
gc.collect()

Using device: cuda
Chemprop matched: d_h 512, depth 3, ffn 512x2, dual readout
Trainable parameters: 1,138,695


1441

In [ ]:
# -----------------------------------------------------------------------------
# 1. Training Loop
# -----------------------------------------------------------------------------
fold_results = []
best_epochs = []
stopped_epochs = []

for fold in FOLDS:
    print(f"\n======[ Fold {fold}/{k_folds} ]======")

    train_dataset, scaler = load_split(fold, "train")
    val_dataset, _ = load_split(fold, "val", scaler)

    train_loader = data.build_dataloader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        pin_memory=True,
        num_workers=NUM_WORKERS,
    )
    val_loader = data.build_dataloader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        pin_memory=True,
        num_workers=NUM_WORKERS,
    )

    model = build_model()
    print(
        f"Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}"
    )
    lightning_model = PlateauChemPropLightningModule(
        model, target_properties, scaler, lr=LR
    )

    save_path_dir = model_path_template.format(fold=fold)
    os.makedirs(save_path_dir, exist_ok=True)

    checkpoint_callback = ModelCheckpoint(
        dirpath=save_path_dir,
        filename="best-{epoch}-{val_loss:.5f}",
        monitor="val_loss",
        mode="min",
        save_top_k=1,
    )
    early_stop = EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=ES_PATIENCE,
        min_delta=0.0,
        check_on_train_epoch_end=False,
        verbose=False,
    )
    best_epoch_tracker = BestEpochTracker()

    logger = TensorBoardLogger(
        save_dir=model_logs_dir, name=model_logs_template.format(fold=fold)
    )

    trainer = pl.Trainer(
        max_epochs=MAX_EPOCHS,
        gradient_clip_val=1.0,
        gradient_clip_algorithm="norm",
        accelerator="auto",
        logger=logger,
        devices=1,
        callbacks=[
            checkpoint_callback,
            early_stop,
            best_epoch_tracker,
            PrintEpochMetricsCallback(target_properties),
        ],
        enable_checkpointing=True,
        num_sanity_val_steps=0,
        log_every_n_steps=1,
        precision="bf16-mixed" if device.type == "cuda" else "32-true",
        enable_progress_bar=False,
    )

    trainer.fit(lightning_model, train_loader, val_loader)

    best_model_score = checkpoint_callback.best_model_score
    if best_model_score is None:
        raise RuntimeError("The checkpoint callback did not record a validation loss.")
    best_val_loss = best_model_score.item()
    best_epoch, last_epoch = best_epoch_tracker.best_epoch, best_epoch_tracker.last_epoch
    if best_epoch is None:
        raise RuntimeError("The best-epoch tracker did not record a validation epoch.")
    best_epochs.append(best_epoch)
    stopped_epochs.append(last_epoch)
    print(f"Best scaled validation loss for fold {fold}: {best_val_loss:.4f}")
    fold_results.append(best_val_loss)

    best_lightning_model = PlateauChemPropLightningModule.load_from_checkpoint(
        checkpoint_callback.best_model_path,
        model=model,
        property_names=target_properties,
        scaler=scaler,
    )
    torch.save(best_lightning_model.model.state_dict(), save_path_dir + "/best.ckpt")

    lightning_model.cpu()
    best_lightning_model.cpu()

    del train_dataset, val_dataset, train_loader, val_loader
    del model, lightning_model, best_lightning_model
    del trainer, checkpoint_callback, logger, early_stop, best_epoch_tracker, scaler

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n======[ Final k-fold results ]======")
print(f"Scaled Loss per fold: {[round(float(x), 4) for x in fold_results]}")
print(f"Average Scaled Loss: {np.mean(fold_results):.4f} ± {np.std(fold_results):.4f}")
print(f"\nBest-validation epoch per fold: {best_epochs} (ran {stopped_epochs})")
print(
    f"Latest best-validation epoch: {max(best_epochs)} of a {MAX_EPOCHS}-epoch ceiling"
)

In [ ]:
# -----------------------------------------------------------------------------
# 2. Evaluation Loop
# -----------------------------------------------------------------------------
import numpy as np

metric_keys = ["mae", "rmsd", "max", "min", "r2"]
all_results = {
    "train": {prop: {m: [] for m in metric_keys} for prop in target_properties},
    "test": {prop: {m: [] for m in metric_keys} for prop in target_properties},
}

for fold in FOLDS:
    print(f"\n======[ Evaluating fold {fold}/{k_folds} ]======")

    train_dset, scaler = load_split(fold, "train")
    test_dset, _ = load_split(fold, "test", scaler)

    train_loader = data.build_dataloader(
        train_dset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        pin_memory=True,
        num_workers=NUM_WORKERS,
    )
    test_loader = data.build_dataloader(
        test_dset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        pin_memory=True,
        num_workers=NUM_WORKERS,
    )

    model = build_model()
    load_path = model_path_template.format(fold=fold) + "/best.ckpt"
    model.load_state_dict(torch.load(load_path, map_location=device))

    lightning_model = PlateauChemPropLightningModule(model, target_properties, scaler)

    trainer = pl.Trainer(
        accelerator="auto", devices=1, logger=False, enable_progress_bar=False
    )

    print("\n======[ Train Set Evaluation ]======")
    train_results = trainer.test(
        lightning_model, dataloaders=train_loader, verbose=False
    )[0]

    print("\n======[ Test Set Evaluation ]======")
    test_results = trainer.test(
        lightning_model, dataloaders=test_loader, verbose=False
    )[0]

    for i, prop in enumerate(target_properties):
        print(f"======[ {prop} ]======")
        for split, res in (("train", train_results), ("test", test_results)):
            mae = res[f"test_mae_{prop}"]
            rmsd = res[f"test_rmsd_{prop}"]
            mx = res[f"test_max_{prop}"]
            mn = res[f"test_min_{prop}"]
            r2 = res[f"test_r2_{prop}"]
            label = "Train" if split == "train" else "Test"
            print(
                f" + {label} -> MAE: {mae:.4f}\tRMSD: {rmsd:.4f}\tMax: {mx:.4f}\tMin: {mn:.4f}\tR^2: {r2:.4f}"
            )
            all_results[split][prop]["mae"].append(mae)
            all_results[split][prop]["rmsd"].append(rmsd)
            all_results[split][prop]["max"].append(mx)
            all_results[split][prop]["min"].append(mn)
            all_results[split][prop]["r2"].append(r2)

    del model, lightning_model, trainer, scaler, train_results, test_results
    del train_loader, test_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"\n======[ Cross-validation results: {TAG} ]======")

ddof = 1 if len(list(FOLDS)) > 1 else 0
for prop in target_properties:
    print(f"\n======[ {prop} ]======")
    for split in ("train", "test"):
        st = {
            m: (
                np.mean(all_results[split][prop][m]),
                np.std(all_results[split][prop][m], ddof=ddof),
            )
            for m in metric_keys
        }
        label = "Train" if split == "train" else "Test"
        print(
            f" + {label} -> "
            f"MAE: {st['mae'][0]:.4f} ± {st['mae'][1]:.4f}\t"
            f"RMSD: {st['rmsd'][0]:.4f} ± {st['rmsd'][1]:.4f}\t"
            f"Max: {st['max'][0]:.4f} ± {st['max'][1]:.4f}\t"
            f"Min: {st['min'][0]:.4f} ± {st['min'][1]:.4f}\t"
            f"R^2: {st['r2'][0]:.4f} ± {st['r2'][1]:.4f}"
        )